# ਪਾਠ 05 - ਏਜੰਟਿਕ RAG


## Setup

This notebook demonstrates the Agentic RAG (Retrieval-Augmented Generation) pattern using the Microsoft Agent Framework.

**Prerequisites:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — your Azure AI Search service endpoint
- `AZURE_SEARCH_API_KEY` — your Azure AI Search API key
- Azure OpenAI deployment configured via environment variables
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ਏਜੰਟਿਕ RAG ਕੀ ਹੈ?

ਪਰੰਪਰাগত RAG ਇੱਕ ਨਿਸ਼ਚਿਤ ਪਾਈਪਲਾਈਨ ਦੀ ਪਾਲਣਾ ਕਰਦਾ ਹੈ: ਦਸਤਾਵੇਜ਼ ਪ੍ਰਾਪਤ ਕਰੋ, ਫਿਰ ਜਵਾਬ ਤਿਆਰ ਕਰੋ। **ਏਜੰਟਿਕ RAG** ਇਸ ਤੋਂ ਅੱਗੇ ਜਾਂਦਾ ਹੈ, ਏਜੰਟ ਨੂੰ ਸੁਤੰਤਰਤਾ ਦਿੰਦਾ ਹੈ ਕਿ ਉਹ ਫੈਸਲਾ ਕਰ ਸਕੇ **ਕਦੋਂ** ਅਤੇ **ਕਿਵੇਂ** ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰਨੀ ਹੈ।

ਏਜੰਟਿਕ RAG ਨਾਲ, ਏਜੰਟ ਕਰ ਸਕਦਾ ਹੈ:
- **ਫੈਸਲਾ ਕਰੋ** ਕਿ ਜਵਾਬ ਦੇਣ ਤੋਂ ਪਹਿਲਾਂ ਪ੍ਰਾਪਤੀ ਦੀ ਲੋੜ ਹੈ ਜਾਂ ਨਹੀਂ
- **ਚੁਣੋ** ਕਿ ਕਿਹੜਾ ਡਾਟਾ ਸਰੋਤ ਜਾਂ ਯੰਤਰ ਪੁੱਛਣਾ ਹੈ
- **ਅੰਕੜਿਆਂ ਦਾ ਮੁਲਾਂਕਣ ਕਰੋ** ਅਤੇ ਪਹਿਲੀ ਕੋਸ਼ਿਸ਼ ਅਪਰਯਾਪਤ ਹੋਣ 'ਤੇ ਅਗਲੀ ਪ੍ਰਾਪਤੀਆਂ ਕਰੋ
- **ਜਾਣਕਾਰੀ ਜੋੜੋ** ਕਈ ਪ੍ਰਾਪਤੀ ਕਦਮਾਂ ਤੋਂ ਮਿਲੀ ਜਾਣਕਾਰੀ ਨੂੰ ਇੱਕ ਸੰਗਠਿਤ ਜਵਾਬ ਵਿੱਚ

ਇਹ ਏਜੰਟ ਨੂੰ ਇੱਕ ਸਥਿਰ ਪ੍ਰਾਪਤ-ਫਿਰ-ਤਿਆਰ ਕਰਨ ਵਾਲੀ ਪਾਈਪਲਾਈਨ ਦੇ ਮੁਕਾਬਲੇ ਹੋਰ ਲਚਕੀਲਾ ਅਤੇ ਸਹੀ ਬਣਾਉਂਦਾ ਹੈ।


## ਇੱਕ ਖੋਜ ਸੰਦ ਬਣਾ ਰਹੇ ਹੋ

ਏਜੰਟਿਕ RAG ਵਿੱਚ, ਬਾਹਰੀ ਡੈਟਾ ਸ੍ਰੋਤਾਂ ਨੂੰ **ਸੰਦਾਂ** ਵਜੋਂ ਪੇਸ਼ ਕੀਤਾ ਜਾਂਦਾ ਹੈ ਜੋ ਏਜੰਟ ਮੰਗ ਹਨ ਤੱਤੇ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ। ਇਸ ਨਾਲ ਏਜੰਟ ਰੀਟਰੀਵਲ ਨੂੰ ਸਿਰਫ ਇੱਕ ਹੋਰ ਕਾਰਵਾਈ ਵਜੋਂ ਲੈਂਦਾ ਹੈ, ਨਾ ਕਿ ਇੱਕ ਜ਼ਰੂਰੀ ਕਦਮ।

ਹੇਠਾਂ ਅਸੀਂ ਇੱਕ ਯਾਤਰਾ ਗਿਆਨ ਅਧਾਰ ਬਣਾਉਂਦੇ ਹਾਂ ਅਤੇ ਇਸਨੂੰ ਸੰਦ ਵਜੋਂ ਪ੍ਰਗਟ ਕਰਦੇ ਹਾਂ, ਜਿਸਨੂੰ ਏਜੰਟ ਮੰਜਿਲ ਦੀ ਜਾਣਕਾਰੀ ਲੱਭਣ ਲਈ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ਏਜੰਟ ਬਣਾਉਣਾ

ਹੁਣ ਅਸੀਂ ਇੱਕ ਏਜੰਟ ਬਣਾਉਂਦੇ ਹਾਂ ਜਿਸਨੂੰ ਹਮੇਸ਼ਾਂ ਉੱਤਰ ਦੇਣ ਤੋਂ ਪਹਿਲਾਂ ਜਾਣਕਾਰੀ **ਲੈਣ ਲਈ ਕਿਹਾ ਗਿਆ ਹੈ**। ਏਜੰਟ ਆਪਣੇ ਜਵਾਬਾਂ ਨੂੰ ਆਪਣੇ ਸਿੱਖਣ ਡੇਟਾ ਦੀ ਬਜਾਏ ਗਿਆਨ ਆਧਾਰ ਵਿੱਚ ਅਡਿੱਠ ਕਰਨ ਲਈ `search_travel_knowledge` ਟੂਲ ਦੀ ਵਰਤੋਂ ਕਰਦਾ ਹੈ।


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ਇਟਰਟਿਵ ਰੀਟਰੀਵਲ — ਮੇਕਰ-ਚੈੱਕਰ ਪੈਟਰਨ

ਏਜੈਂਟਿਕ RAG ਦਾ ਇੱਕ ਮੁੱਖ ਲਾਭ **ਇਟਰਟਿਵ ਰੀਟਰੀਵਲ** ਹੈ। ਏਜੈਂਟ ਆਪਣੀਆਂ ਸ਼ੁਰੂਆਤੀ ਖੋਜਾਂ ਦੀ ਪੁਸ਼ਟੀ ਕਰਨ, ਸੁਧਾਰ ਕਰਨ ਜਾਂ ਵਿਸਥਾਰ ਕਰਨ ਲਈ ਕਈ ਦੌਰਾਂ ਦੀ ਖੋਜ ਕਰ ਸਕਦਾ ਹੈ — ਇਕ "ਮੇਕਰ-ਚੈੱਕਰ" ਵਰਕਫਲੋਅ ਵਾਂਗ:

1. **ਮੇਕਰ ਕਦਮ**: ਏਜੈਂਟ ਮੁਢਲੀ ਜਾਣਕਾਰੀ ਪ੍ਰਾਪਤ ਕਰਦਾ ਹੈ ਅਤੇ ਜਵਾਬ ਦਾ ਡ੍ਰਾਫਟ ਤਿਆਰ ਕਰਦਾ ਹੈ।
2. **ਚੈੱਕਰ ਕਦਮ**: ਏਜੈਂਟ ਵਾਧੂ ਰੀਟਰੀਵਲ ਕਰਦਾ ਹੈ ਤਾਂ ਜੋ ਵੇਰਵੇ ਦੀ ਪੁਸ਼ਟੀ ਕਰੇ ਜਾਂ ਖਾਮੀਆਂ ਭਰੇ।

ਹੇਠਾਂ, ਏਜੈਂਟ ਨੂੰ ਇੱਕ ਸਵਾਲ ਪੁੱਛਿਆ ਗਿਆ ਹੈ ਜਿਸ ਵਿੱਚ ਕਈ ਮੰਜ਼ਿਲਾਂ ਦੀ ਤੁਲਨਾ ਕਰਨ ਦੀ ਲੋੜ ਹੈ, ਜਿਸ ਨਾਲ ਉਹ ਕਈ ਵਾਰੀ ਖੋਜ ਕਰਨ ਲਈ ਮਨਾਇਆ ਜਾਂਦਾ ਹੈ।


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## ਸੰਖੇਪ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ ਕਿ ਕਿਵੇਂ ਮਾਈਕਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਦੀ ਵਰਤੋਂ ਕਰਕੇ **Agentic RAG** ਸਿਸਟਮ ਬਣਾਇਆ ਜਾ ਸਕਦਾ ਹੈ:

- **Agentic RAG** ਏਜੰਟਾਂ ਨੂੰ ਆਪਣੀ ਭਰੋਸੇਯੋਗ ਜਾਣਕਾਰੀ ਲੈਣ ਦਾ ਫੈਸਲਾ ਖੁਦ ਕਰਨ ਦਿੰਦਾ ਹੈ, ਜਿਸ ਨਾਲ ਜਾਣਕਾਰੀ ਲੈਣਾ ਨਿਸਚਿਤ ਨਾ ਹੋਕੇ ਗਤੀਸ਼ੀਲ ਹੁੰਦਾ ਹੈ।
- **ਟੂਲਾਂ ਦੇ ਤੌਰ 'ਤੇ ਡੇਟਾ ਸਰੋਤ**: ਬਾਹਰੀ ਗਿਆਨ ਆਧਾਰ (ਜਿਵੇਂ Azure AI Search) ਨੂੰ ਟੂਲਾਂ ਵਜੋਂ ਪੇਸ਼ ਕੀਤਾ ਜਾਂਦਾ ਹੈ ਜਿਨ੍ਹਾਂ ਨੂੰ ਏਜੰਟ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।
- **ਪੁਨਰਾਵਰਤੀ ਜਾਣਕਾਰੀ ਲੈਣਾ**: maker-checker ਪੈਟਰਨ ਏਜੰਟ ਨੂੰ ਕਈ ਵਾਰ ਜਾਣਕਾਰੀ ਲੈਣ ਦੀ ਆਗਿਆ ਦਿੰਦਾ ਹੈ — ਖੋਜ, ਤਸਦੀਕ ਅਤੇ ਸੁਧਾਰ ਕਰਨ — ਫਿਰ ਅੰਤਿਮ ਉੱਤਰ ਦੇਣ ਤੋਂ ਪਹਿਲਾਂ।

ਉਤਪਾਦਨ ਵਿੱਚ, ਤੁਸੀਂ in-memory `TRAVEL_KNOWLEDGE_BASE` ਦੀ ਥਾਂ ਅਸਲ Azure AI Search ਇੰਡੈਕਸ ਰੱਖੋਗੇ ਤਾਂ ਜੋ ਵੱਡੇ ਪੱਧਰ ਤੇ ਯਾਤਰਾ ਦਸਤਾਵੇਜ਼ਾਂ ਦੀ ਖੋਜ ਸੰਭਾਲੀ ਜਾ ਸਕੇ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
